## Imports

In [95]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests


## Globale Variablen

In [96]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig."

LOCAL = True;

## Tools

Retreival Tool

In [98]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description='Retrieves information from Lecture related Documents')
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()

## Initialisierungen

In [99]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url="http://192.168.178.125:11434",
    model="gpt-oss:20b",
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[get_system_info, search_lecture_docs], system_prompt=system_prompt)

In [100]:
prompt = str(input())
result = agent.invoke({"messages": [("user", prompt)]})
print(result["messages"][-1].content)

Ja – als Mitarbeiter kannst du dich fachlich weiterbilden.  
In den bereitgestellten Unterlagen steht, dass Mitarbeitende jährlich **bis zu 1 000 €** für fachliche Weiterbildung beantragen dürfen. Damit kannst du z. B. Fortbildungen, Seminare oder ähnliche Qualifizierungsmaßnahmen finanzieren lassen.  

Falls du konkretere Fragen hast (z. B. zu Antragsformalitäten oder Genehmigungsverfahren), sag einfach Bescheid!
